In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel('../data/processed/processed_first_task.xlsx')
df = df.drop(columns = 'Unnamed: 0')
df = df.dropna()
df.head()

,symptoms,Angina range,depression,ECG range,WMSI stress,D WMSI,WMSI range,target,log_WMSI_stress,WMSI_stress_high
0,3,0,3,0,1.000,0.000,0,1.0,0.000000,0
1,3,0,3,0,1.125,0.000,0,1.0,0.117783,1
2,3,0,3,0,1.125,0.125,1,0.0,0.117783,1
3,1,1,3,0,1.000,-0.190,1,1.0,0.000000,0
4,3,0,3,0,1.120,0.120,1,0.0,0.113329,0


In [3]:
X = df[['symptoms', 'depression', 'log_WMSI_stress', 'D WMSI']]
y = df['target']

In [4]:
from catboost import CatBoostClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    make_scorer
)

features = [
    'symptoms',
    'depression',
    'log_WMSI_stress',
    'D WMSI'
]

X = df[features].copy()
y = df['target'].copy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

specificity_scorer = make_scorer(specificity_score)

model_cat = CatBoostClassifier(
    iterations=300,
    depth=4,
    learning_rate=0.03,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=0,
    auto_class_weights='Balanced'
)

scoring = {
    "f1": make_scorer(f1_score),
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "accuracy": make_scorer(accuracy_score),
    "specificity": specificity_scorer,
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    model_cat,
    X,
    y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

metrics_df = pd.DataFrame({
    "F1 mean": [cv_results["test_f1"].mean()],
    "F1 std": [cv_results["test_f1"].std()],

    "Precision mean": [cv_results["test_precision"].mean()],
    "Precision std": [cv_results["test_precision"].std()],

    "Recall mean": [cv_results["test_recall"].mean()],
    "Recall std": [cv_results["test_recall"].std()],

    "Specificity mean": [cv_results["test_specificity"].mean()],
    "Specificity std": [cv_results["test_specificity"].std()],

    "Accuracy mean": [cv_results["test_accuracy"].mean()],
    "Accuracy std": [cv_results["test_accuracy"].std()],

    "ROC-AUC mean": [cv_results["test_roc_auc"].mean()],
    "ROC-AUC std": [cv_results["test_roc_auc"].std()],
})

print(metrics_df)
model_cat.fit(X, y)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": model_cat.get_feature_importance()
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("\nFeature importance:")
print(importance_df)

y_proba = model_cat.predict_proba(X)[:, 1]

auc = roc_auc_score(y, y_proba)

print(f"\nROC-AUC on full data: {auc:.4f}")

   F1 mean  F1 std  Precision mean  Precision std  Recall mean  Recall std  \
0  0.47853  0.0889        0.444524       0.082936     0.524786    0.114474   

   Specificity mean  Specificity std  Accuracy mean  Accuracy std  \
0          0.761459         0.052679       0.699248       0.04982   

   ROC-AUC mean  ROC-AUC std  
0      0.649629      0.05341  

Feature importance:
           feature  importance
3           D WMSI   42.760180
2  log_WMSI_stress   29.156510
1       depression   14.215891
0         symptoms   13.867419

ROC-AUC on full data: 0.7953


In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score, StratifiedKFold


def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-5, 10, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 1e-5, 10, log=True),
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': 0,
        'auto_class_weights': 'Balanced'
    }
    
    # if trial.suggest_categorical('use_subsample', [True, False]):
    #     params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
    
    # if trial.suggest_categorical('use_one_hot', [True, False]):
    #     params['one_hot_max_size'] = trial.suggest_int('one_hot_max_size', 2, 10)
    
    model = CatBoostClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring_metric = 'roc_auc' 
    
    if scoring_metric == 'specificity':
        scorer = make_scorer(specificity_score)
        scores = cross_val_score(model, X, y, cv=cv, scoring=scorer, n_jobs=-1)
    elif scoring_metric == 'f1':
        scores = cross_val_score(model, X, y, cv=cv, scoring='f1', n_jobs=-1)
    elif scoring_metric == 'roc_auc':
        scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    elif scoring_metric == 'accuracy':
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    elif scoring_metric == 'recall':
        scores = cross_val_score(model, X, y, cv=cv, scoring='recall', n_jobs=-1)
    elif scoring_metric == 'precision':
        scores = cross_val_score(model, X, y, cv=cv, scoring='precision', n_jobs=-1)
    else:
        scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    return scores.mean()

study = optuna.create_study(
    direction='maximize',  
    sampler=TPESampler(seed=42),
    study_name='catboost_optimization'
)

study.optimize(
    objective, 
    n_trials=50, 
    n_jobs=1,
    show_progress_bar=True
)

print("ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"{key}: {value}")

print(f"\nЛучшее значение метрики: {study.best_value:.4f}")


optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)
optuna.visualization.plot_parallel_coordinate(study)


final_params = best_params.copy()
final_params.update({
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 0,
    'auto_class_weights': 'Balanced'
})

print("Параметры модели:")
for key, value in final_params.items():
    print(f"  {key}: {value}")

final_model = CatBoostClassifier(**final_params)
final_model.fit(X, y)

from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

scoring = {
    "f1": make_scorer(f1_score),
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "accuracy": make_scorer(accuracy_score),
    "specificity": make_scorer(specificity_score),
    "roc_auc": "roc_auc"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    final_model,
    X,
    y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

# Результаты
metrics_df = pd.DataFrame({
    "F1 mean": [cv_results["test_f1"].mean()],
    "F1 std": [cv_results["test_f1"].std()],
    "Precision mean": [cv_results["test_precision"].mean()],
    "Precision std": [cv_results["test_precision"].std()],
    "Recall mean": [cv_results["test_recall"].mean()],
    "Recall std": [cv_results["test_recall"].std()],
    "Specificity mean": [cv_results["test_specificity"].mean()],
    "Specificity std": [cv_results["test_specificity"].std()],
    "Accuracy mean": [cv_results["test_accuracy"].mean()],
    "Accuracy std": [cv_results["test_accuracy"].std()],
    "ROC-AUC mean": [cv_results["test_roc_auc"].mean()],
    "ROC-AUC std": [cv_results["test_roc_auc"].std()],
})

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ФИНАЛЬНОЙ МОДЕЛИ (CV):")
print("="*50)
print(metrics_df)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": final_model.get_feature_importance()
}).sort_values(by="importance", ascending=False)

print("\n" + "="*50)
print("FEATURE IMPORTANCE:")
print("="*50)
print(importance_df)


y_proba = final_model.predict_proba(X)[:, 1]
auc = roc_auc_score(y, y_proba)
print(f"\nROC-AUC on full data: {auc:.4f}")

[I 2026-05-17 17:13:48,622] A new study created in memory with name: catboost_optimization


Начинаем оптимизацию гиперпараметров...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-17 17:13:51,662] Trial 0 finished with value: 0.5354576388822964 and parameters: {'iterations': 450, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 0.039079671568228794, 'border_count': 66, 'bagging_temperature': 0.15599452033620265, 'random_strength': 2.231010801867923e-05}. Best is trial 0 with value: 0.5354576388822964.
[I 2026-05-17 17:13:55,195] Trial 1 finished with value: 0.523173225912952 and parameters: {'iterations': 900, 'depth': 7, 'learning_rate': 0.11114989443094977, 'l2_leaf_reg': 1.3289448722869181e-05, 'border_count': 249, 'bagging_temperature': 0.8324426408004217, 'random_strength': 0.00018794668241638458}. Best is trial 0 with value: 0.5354576388822964.
[I 2026-05-17 17:13:55,737] Trial 2 finished with value: 0.5859158948200044 and parameters: {'iterations': 250, 'depth': 4, 'learning_rate': 0.028145092716060652, 'l2_leaf_reg': 0.014077923139972392, 'border_count': 128, 'bagging_temperature': 0.2912291401980419, 'random_strength': 0.04689